# Hafta 06: Veri Bağı Katmanı (Link Layer) — Çerçeveleme, MAC Adresleme, ARP ve Anahtarlama (Switch)

> **Ders Kitabı Referansı:** Computer Networking: A Top-Down Approach (J.F. Kurose, K.W. Ross — Bölüm 5)

---

## 1. Temel Kavramlar ve Mimari Çerçeve

Ağ katmanı (IP) paketleri küresel internette **kaynak makineden hedef makineye** yönlendirirken; **Veri Bağı Katmanı (Link Layer)**, bu datagramların fiziksel kablo, fiber veya Wi-Fi üzerinden **birbirine doğrudan bağlı iki komşu düğüm arasında (Node-to-Node)** güvenle sıçramasını sağlar.

---

### 1.1. Temel Link Katmanı Servisleri

1. **Çerçeveleme (Framing):** Ağ katmanından gelen IP datagramını alır, başına **Başlık (Header: Kaynak/Hedef MAC, EtherType)** ve sonuna **Kuyruk (Trailer: CRC/FCS Hata Denetimi)** ekleyerek bir **Çerçeveye (Frame)** dönüştürür.
2. **Ortama Erişim Kontrolü (Medium Access Control - MAC):**
   * **Noktadan Noktaya Linkler:** Tek bir gönderici ve tek bir alıcı varsa (örneğin iki router arası kiralık fiber hat) koordinasyona gerek yoktur.
   * **Paylaşımlı Ortamlar (Broadcast / Shared Media):** Wi-Fi veya eski koaksiyel Ethernet hatlarında aynı ortamı yüzlerce cihaz paylaşır. İki cihaz aynı anda sinyal basarsa çarpışma (*Collision*) oluşur. MAC kuralları (CSMA/CD veya CSMA/CA), iletim öncesi hattı dinleyerek çakışmaları engeller.

---

### 1.2. Neden Hem IP Hem MAC Adresine İhtiyaç Vardır?

| Karşılaştırma Ölçütü | MAC Adresi (Fiziksel / Donanım) | IP Adresi (Mantıksal / Ağ) |
| :--- | :--- | :--- |
| **Boyut ve Format** | **48 bit (6 Bayt)**, Hex: `AA:BB:CC:DD:EE:FF` | **32 bit (4 Bayt)**, Noktalı: `192.168.1.1` |
| **Yapı** | **Düz (Flat):** Coğrafi veya hiyerarşik anlam taşımaz. | **Hiyerarşik:** Ağ öneki (Prefix) konumu bildirir. |
| **Değişebilirlik** | Ağ kartı üretilirken ROM'a kazınır (*Burned-in*), sabittir. | Bağlanılan ağa göre dinamik değişir (DHCP). |
| **İşlev Alanı** | **Aynı yerel ağ (LAN / Alt Ağ)** içinde komşuyu bulur. | **Farklı ağlar ve küresel internette** yönlendirir. |
| **Analoji** | Bir insanın **T.C. Kimlik Numarası** gibidir (Hiç değişmez). | Bir insanın **Posta Adresi** gibidir (Taşındıkça değişir). |

> **Kritik Gerçek:** Tüm dünya yalnızca MAC adresleriyle çalışamazdı. 48-bitlik düz bir adreste konum bilgisi olmadığı için, dünyanın tüm yönlendiricilerinin 281 trilyon cihazın tamamını tek tek ezberinde tutması gerekirdi. Bu nedenle internet hiyerarşik **IP** adresleriyle yönlendirilir; yerel alt ağa inildiğinde ise fiziksel hatta veriyi teslim etmek için **MAC** kullanılır.
>
> **SORU: Evinizdeki modemin internet kablosunu çekseniz (dış internet kesilse), evdeki iki bilgisayar birbiriyle dosya paylaşabilir veya yerel yazıcıdan çıktı alabilir mi?**  
> **CEVAP:** Evet, kesinlikle çalışır! Switch Katman 2 cihazıdır; IP adreslerini veya dış interneti hiç bilmez. Cihazlar yerel ağda birbirine doğrudan **MAC adresleriyle** çerçeve fırlattığı için internet bağlantısı sağlayan genel bir IP olmasa dahi yerel LAN trafiği kusursuz işlemeye devam eder.

---

### 1.3. Adres Çözümleme Protokolü (ARP - Address Resolution Protocol)

Bir cihazın elinde gideceği hedef makinenin IP adresi vardır; ancak o makineye Ethernet çerçevesi basabilmek için hedef makinenin **MAC adresini** bilmek zorundadır. Bu iki adres arasındaki köprüyü **ARP (Address Resolution Protocol)** kurar:

* **ARP İsteği (ARP Request - Broadcast):**
  * Gönderici yerel ağdaki herkese sorar: *"Kimde 192.168.1.1 IP'si var? Varsa lütfen bana MAC adresini söylesin!"*
  * Hedef MAC adresi **`FF:FF:FF:FF:FF:FF`** (Broadcast) yapılır. Switch bu çerçeveyi gelen port hariç yerel ağdaki tüm cihazlara basar.
* **ARP Yanıtı (ARP Reply - Unicast):**
  * O IP'ye sahip olan cihaz seslenir: *"O IP bende, fiziksel MAC adresim 00:11:22:33:44:55!"*
  * Yanıt doğrudan soran cihazın MAC adresine özel (*Unicast*) döner.
* **ARP Tablosu (ARP Cache):** Cihazlar her paket öncesi ağa ARP fırlatıp hattı tıkamamak için öğrendikleri eşleşmeleri işletim sistemi RAM'inde saklar (`IP <-> MAC <-> TTL`).

> **DNS vs ARP vs RARP Ayrımı:**  
> * **DNS:** Küresel internette insanların okuduğu Alan Adını IP'ye çevirir (`www.cbu.edu.tr -> 194.27.156.248`).  
> * **ARP:** Yalnızca aynı alt ağ (LAN) içinde mantıksal IP adresini fiziksel MAC donanım adresine çevirir (`192.168.1.1 -> 00:11:22:33:44:55`).  
> * **RARP (Reverse ARP):** ARP mantığının tam tersine çalışarak, kendi IP adresini henüz bilmeyen disksiz bir makinenin kendi MAC adresini anons edip ağdan IP adresi talep etmesini sağlar (Modern ağlarda yerini DHCP'ye bırakmıştır).

---
## 2. Mimari Görselleştirmeler (Vektörel SVG)

Aşağıdaki kod hücreleri; Standart Ethernet II Çerçeve Anatomisini (Header, Payload, CRC) ve Yerel Ağda ARP Sorgu/Cevap Döngüsünü DLL bağımsız saf SVG formatında ekrana basar.

In [ ]:
from IPython.display import SVG, display

# Şekil 1: Standart Ethernet II Çerçeve Anatomisi (Frame Format)
svg_frame = """
<svg width="820" height="280" viewBox="0 0 820 280" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">
    Standart Ethernet II Çerçeve Yapısı (Frame Anatomy)
  </text>

  <g transform="translate(30, 50)">
    <rect width="760" height="180" rx="8" fill="#f8fafc" stroke="#64748b" stroke-width="1.5"/>
    
    <!-- Preamble -->
    <rect x="15" y="35" width="95" height="75" rx="4" fill="#f1f5f9" stroke="#94a3b8" stroke-width="1" stroke-dasharray="3,3"/>
    <text x="62" y="68" text-anchor="middle" font-size="10" font-weight="bold" fill="#475569">Öncü Bitler</text>
    <text x="62" y="85" text-anchor="middle" font-size="9" fill="#64748b">Preamble (8B)</text>
    <text x="62" y="100" text-anchor="middle" font-size="8" fill="#94a3b8">Saat Senkronu</text>

    <!-- Dest MAC -->
    <rect x="115" y="35" width="130" height="75" rx="4" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="180" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">Hedef MAC</text>
    <text x="180" y="85" text-anchor="middle" font-size="9" fill="#1d4ed8">Dest MAC (6 Bayt)</text>
    <text x="180" y="100" text-anchor="middle" font-size="8" fill="#1e40af">48-bit Donanım</text>

    <!-- Src MAC -->
    <rect x="250" y="35" width="130" height="75" rx="4" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="315" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">Kaynak MAC</text>
    <text x="315" y="85" text-anchor="middle" font-size="9" fill="#1d4ed8">Source MAC (6B)</text>
    <text x="315" y="100" text-anchor="middle" font-size="8" fill="#1e40af">Kartın Kendi MAC'i</text>

    <!-- Type -->
    <rect x="385" y="35" width="85" height="75" rx="4" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
    <text x="427" y="65" text-anchor="middle" font-size="10" font-weight="bold" fill="#92400e">Tür</text>
    <text x="427" y="80" text-anchor="middle" font-size="9" fill="#b45309">Type (2B)</text>
    <text x="427" y="95" text-anchor="middle" font-size="7" font-weight="bold" fill="#78350f">0800: IPv4</text>
    <text x="427" y="105" text-anchor="middle" font-size="7" font-weight="bold" fill="#78350f">0806: ARP</text>

    <!-- Payload -->
    <rect x="475" y="35" width="175" height="75" rx="4" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
    <text x="562" y="68" text-anchor="middle" font-size="11" font-weight="bold" fill="#14532d">Veri Yükü (Payload)</text>
    <text x="562" y="85" text-anchor="middle" font-size="9" fill="#15803d">46 - 1500 Bayt (MTU)</text>
    <text x="562" y="100" text-anchor="middle" font-size="8" fill="#166534">IP Datagramı veya ARP Yükü</text>

    <!-- CRC / FCS -->
    <rect x="655" y="35" width="90" height="75" rx="4" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/>
    <text x="700" y="68" text-anchor="middle" font-size="10" font-weight="bold" fill="#991b1b">CRC / FCS</text>
    <text x="700" y="85" text-anchor="middle" font-size="9" fill="#b91c1c">Hata Kodu (4B)</text>
    <text x="700" y="100" text-anchor="middle" font-size="8" fill="#7f1d1d">Donanımsal Denetim</text>

    <!-- Alt Açıklama -->
    <text x="380" y="145" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e293b">
      Toplam Protokol Başlığı: 14 Bayt (MAC'ler + Type) + 4 Bayt Kuyruk (FCS) = 18 Bayt Ek Yük
    </text>
    <text x="380" y="165" text-anchor="middle" font-size="10" fill="#64748b" font-style="italic">
      Eğer ağ katmanı yükü 46 bayttan küçükse, Ethernet standardı gereği çerçeve sonuna sıfırlar (Padding) doldurulur.
    </text>
  </g>
</svg>
"""
display(SVG(svg_frame))

In [ ]:
# Şekil 2: Yerel Ağda ARP Çözümleme (Broadcast İstek vs Unicast Yanıt)
svg_arp = """
<svg width="820" height="320" viewBox="0 0 820 320" xmlns="http://www.w3.org/2000/svg" style="background:#ffffff; font-family:sans-serif;">
  <defs>
    <marker id="arr-arp-b" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#2563eb"/>
    </marker>
    <marker id="arr-arp-g" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0, 7 3.5, 0 7" fill="#16a34a"/>
    </marker>
  </defs>
  <text x="410" y="24" text-anchor="middle" font-size="14" font-weight="bold" fill="#111827">
    ARP Çözümleme Akışı: Broadcast Sorgusu ve Unicast Yanıtı
  </text>

  <!-- Düğüm 1: Host A -->
  <g transform="translate(60, 60)">
    <rect width="150" height="75" rx="6" fill="#dbeafe" stroke="#2563eb" stroke-width="1.5"/>
    <text x="75" y="25" text-anchor="middle" font-size="11" font-weight="bold" fill="#1e40af">HOST A (İsteyen)</text>
    <text x="75" y="45" text-anchor="middle" font-size="9" fill="#1d4ed8">IP: 192.168.1.10</text>
    <text x="75" y="62" text-anchor="middle" font-size="9" fill="#1e3a8a">MAC: AA:AA:AA:AA:AA:AA</text>
  </g>

  <!-- Merkez: L2 Switch -->
  <g transform="translate(340, 110)">
    <rect width="140" height="85" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="2"/>
    <text x="70" y="26" text-anchor="middle" font-size="11" font-weight="bold" fill="#92400e">KATMAN 2 SWITCH</text>
    <text x="70" y="44" text-anchor="middle" font-size="9" fill="#b45309">MAC Öğrenme Tablosu</text>
    <line x1="15" y1="52" x2="125" y2="52" stroke="#fde68a" stroke-width="1"/>
    <text x="70" y="68" text-anchor="middle" font-size="8" fill="#78350f">Port 1: AA.. | Port 2: BB..</text>
  </g>

  <!-- Düğüm 2: Host B -->
  <g transform="translate(610, 60)">
    <rect width="150" height="75" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
    <text x="75" y="25" text-anchor="middle" font-size="11" font-weight="bold" fill="#14532d">HOST B (Hedef)</text>
    <text x="75" y="45" text-anchor="middle" font-size="9" fill="#15803d">IP: 192.168.1.20</text>
    <text x="75" y="62" text-anchor="middle" font-size="9" fill="#166534">MAC: BB:BB:BB:BB:BB:BB</text>
  </g>

  <!-- Düğüm 3: Host C (İlgisiz Cihaz) -->
  <g transform="translate(610, 190)">
    <rect width="150" height="75" rx="6" fill="#f1f5f9" stroke="#94a3b8" stroke-width="1.5" stroke-dasharray="3,3"/>
    <text x="75" y="25" text-anchor="middle" font-size="11" font-weight="bold" fill="#475569">HOST C (İlgisiz)</text>
    <text x="75" y="45" text-anchor="middle" font-size="9" fill="#64748b">IP: 192.168.1.30</text>
    <text x="75" y="62" text-anchor="middle" font-size="9" fill="#64748b">MAC: CC:CC:CC:CC:CC:CC</text>
  </g>

  <!-- 1. Adım: ARP REQUEST (BROADCAST) -->
  <line x1="210" y1="95" x2="330" y2="130" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-arp-b)"/>
  <line x1="480" y1="130" x2="600" y2="95" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-arp-b)"/>
  <line x1="480" y1="160" x2="600" y2="210" stroke="#2563eb" stroke-width="2" marker-end="url(#arr-arp-b)"/>
  <text x="260" y="85" font-size="9" font-weight="bold" fill="#1e40af">1. ARP İstek (Broadcast)</text>
  <text x="545" y="240" font-size="8" fill="#64748b">Host C bakar, IP eşleşmez -&gt; DROP!</text>

  <!-- 2. Adım: ARP REPLY (UNICAST) -->
  <line x1="600" y1="115" x2="485" y2="140" stroke="#16a34a" stroke-width="2" stroke-dasharray="4,3" marker-end="url(#arr-arp-g)"/>
  <line x1="340" y1="155" x2="215" y2="120" stroke="#16a34a" stroke-width="2" stroke-dasharray="4,3" marker-end="url(#arr-arp-g)"/>
  <text x="260" y="160" font-size="9" font-weight="bold" fill="#15803d">2. ARP Yanıt (Unicast)</text>

  <text x="410" y="295" text-anchor="middle" font-size="10" fill="#334155">
    Sorgu: dst_mac = FF:FF:FF:FF:FF:FF | Yanıt: dst_mac = AA:AA:AA:AA:AA:AA, "192.168.1.20 bendedir"
  </text>
</svg>
"""
display(SVG(svg_arp))

---
## 3. Deney 1: Ethernet Çerçeve İnşası ve CRC32 / FCS Hata Denetim Simülatörü

### 3.1. Deneyin Amacı
Kurose & Ross Bölüm 5.1'de anlatılan çerçeveleme (*Framing*) ve hata yakalama (*Error Detection*) mekanizmasını modellemek; donanım seviyesinde çalışan **CRC-32 (Cyclic Redundancy Check)** algoritması ile çerçevenin kuyruğuna (*Trailer / FCS*) eklenen sağlama kodunun, kablodaki tek bitlik bir elektriksel bozulmada dahi çerçeveyi nasıl anında çöpe attığını (*Discard / Drop*) kanıtlamaktır.

In [ ]:
# ==============================================================================
# DENEY 1: Ethernet Çerçevesi İnşası ve CRC-32 (FCS) Hata Denetimi
# ==============================================================================
import binascii
import struct

def mac_to_bytes(mac_str: str) -> bytes:
    """'AA:BB:CC:DD:EE:FF' şeklindeki metni 6 baytlık ikili formata çevirir"""
    return bytes.fromhex(mac_str.replace(":", ""))

def build_ethernet_frame(src_mac: str, dst_mac: str, ethertype: int, payload: bytes) -> bytes:
    """
    Ethernet II Çerçevesi Oluşturur: Dest MAC (6B) + Src MAC (6B) + Type (2B) + Payload + CRC (4B)
    """
    header = mac_to_bytes(dst_mac) + mac_to_bytes(src_mac) + struct.pack("!H", ethertype)
    
    # Minimum Ethernet yükü 46 bayt olmalıdır; küçükse Padding eklenir
    if len(payload) < 46:
        payload = payload + (b"\x00" * (46 - len(payload)))
        
    frame_without_fcs = header + payload
    # Donanımsal CRC-32 (Frame Check Sequence - FCS) hesaplanır
    crc32_val = binascii.crc32(frame_without_fcs) & 0xFFFFFFFF
    fcs_bytes = struct.pack("!I", crc32_val)
    
    return frame_without_fcs + fcs_bytes

def verify_and_parse_frame(raw_frame: bytes):
    """Alıcı Ağ Kartı (NIC) Ayrıştırma ve Hata Denetim Fonksiyonu"""
    data_part = raw_frame[:-4]
    received_fcs = struct.unpack("!I", raw_frame[-4:])[0]
    calculated_fcs = binascii.crc32(data_part) & 0xFFFFFFFF
    
    if received_fcs != calculated_fcs:
        return False, "[DONANIM HATASI] CRC Uyuşmazlığı! Çerçeve kabloda bozulmuş, anında DROP edildi."
        
    dst_mac = ":".join(f"{b:02x}" for b in data_part[0:6])
    src_mac = ":".join(f"{b:02x}" for b in data_part[6:12])
    ethertype = struct.unpack("!H", data_part[12:14])[0]
    payload = data_part[14:]
    
    return True, {
        "dst_mac": dst_mac,
        "src_mac": src_mac,
        "ethertype": f"0x{ethertype:04X}",
        "payload_len": len(payload),
        "fcs": f"0x{received_fcs:08X}"
    }

# --- TESTLER ---
print("=" * 65)
print("ETHERNET II ÇERÇEVE İNŞASI VE CRC-32 SAĞLAMA ANALİZİ")
print("=" * 65)

# 1. Senaryo: Hatasız Gönderim
orijinal_cerceve = build_ethernet_frame(
    src_mac="AA:BB:CC:11:22:33",
    dst_mac="FF:EE:DD:44:55:66",
    ethertype=0x0800,  # IPv4
    payload=b"IP_DATAGRAM_ICERIGI_TEST_VERISI"
)
gecerli, sonuc = verify_and_parse_frame(orijinal_cerceve)
print(f"[SENARYO 1 - HATASIZ ÇERÇEVE] : {'KABUL EDİLDİ' if gecerli else 'RED'}")
print(f"  - Hedef MAC     : {sonuc['dst_mac']}")
print(f"  - Kaynak MAC    : {sonuc['src_mac']}")
print(f"  - EtherType     : {sonuc['ethertype']} (IPv4 Protokolü)")
print(f"  - Toplam Boyut  : {len(orijinal_cerceve)} Bayt (14B Başlık + {sonuc['payload_len']}B Yük + 4B FCS)")
print(f"  - FCS Değeri    : {sonuc['fcs']}")

# 2. Senaryo: Kabloda 1 Bit Değişirse (Gürültü)
bozuk_cerceve = bytearray(orijinal_cerceve)
bozuk_cerceve[15] ^= 0x01  # Yükteki 1 biti ters çeviriyoruz
gecerli_2, sonuc_2 = verify_and_parse_frame(bytes(bozuk_cerceve))
print(f"\n[SENARYO 2 - 1-BIT BOZULMA]  : {sonuc_2}")
print("=" * 65)

---
### 3.2. Deney 1'in Mühendislik Yorumu
1. **Asgari Boyut (Minimum Frame Size - 64 Bayt):** Ethernet standardında bir çerçeve başlık ve FCS dahil en az **64 bayt** olmak zorundadır. Bu sebeple veri yükü 46 baytın altındaysa araya otomatik olarak sıfırlar (*Padding*) eklenir. Bunun temel gerekçesi, paylaşımlı hatta çarpışma algılama (*CSMA/CD collision detection*) süresinin garanti altına alınmasıdır.
2. **Ağ Kartı (NIC) Düzeyinde Sessiz İmha:** CRC kontrolü işletim sistemi çekirdeğine (CPU'ya) yük bindirmez; doğrudan ağ adaptörü donanımında yapılır. Hatalı çerçeveler üst katmanlara hiç iletilmeden donanım seviyesinde atılır.
3. **Promiscuous (Karma) Mod vs Normal Filtreleme:** Normal şartlarda ağ kartı (NIC), gelen çerçevenin hedef MAC adresine bakar; hedef kendi MAC'i veya `FF:FF:FF:FF:FF:FF` (Broadcast) değilse işlemciye hiç iletmeden çöpe atar (*Discard*). Ancak Wireshark gibi bir paket koklama (*Sniffing*) aracı çalıştırıldığında ağ kartı **Promiscuous Moda** geçer ve hattan geçen başkalarına ait tüm çerçeveleri işletim sistemine aktarır.

---
## 4. Deney 2: Scapy ile ARP İsteği ve Yanıtı Paketleme + .pcap Analizi

### 4.1. Deneyin Amacı
Yerel ağda bir makinenin IP adresinden fiziksel MAC adresini keşfetmek için atılan **ARP İsteği (Broadcast)** ve dönen **ARP Yanıtını (Unicast)** Python `scapy` ile inşa etmek; başlık alanlarındaki Operasyon Kodunu (`op=1` Request, `op=2` Reply) ve donanım adreslerini Wireshark `.pcap` dosyasında ayrıştırmaktır.

In [ ]:
# ==============================================================================
# DENEY 2: Scapy ile ARP İstek ve Yanıt Paketlerinin İnşası
# ==============================================================================
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from scapy.all import Ether, ARP, wrpcap, rdpcap

HOST_A_IP = "192.168.1.10"
HOST_A_MAC = "00:11:22:33:44:aa"

GATEWAY_IP = "192.168.1.1"
GATEWAY_MAC = "00:11:22:33:44:01"

# 1. Adım: ARP İsteği (Host A soruyor: "192.168.1.1 kimde? Bana söyle")
# Ethernet Hedef MAC: Broadcast (ff:ff:ff:ff:ff:ff) | ARP op=1 (who-has)
arp_request = (
    Ether(src=HOST_A_MAC, dst="ff:ff:ff:ff:ff:ff") /
    ARP(
        op=1,                 # 1 = ARP Request
        hwsrc=HOST_A_MAC,     # Benim MAC
        psrc=HOST_A_IP,       # Benim IP
        hwdst="00:00:00:00:00:00", # Henüz bilinmiyor!
        pdst=GATEWAY_IP       # Aradığım IP
    )
)

# 2. Adım: ARP Yanıtı (Gateway cevaplıyor: "192.168.1.1 bendedir, MAC adresim budur")
# Ethernet Hedef MAC: Doğrudan Host A (Unicast) | ARP op=2 (is-at)
arp_reply = (
    Ether(src=GATEWAY_MAC, dst=HOST_A_MAC) /
    ARP(
        op=2,                 # 2 = ARP Reply
        hwsrc=GATEWAY_MAC,    # Cevap veren MAC
        psrc=GATEWAY_IP,      # Cevap veren IP
        hwdst=HOST_A_MAC,     # İsteyen MAC
        pdst=HOST_A_IP        # İsteyen IP
    )
)

# Paketleri .pcap dosyasına yazalım
pcap_file = "arp_cozumleme.pcap"
wrpcap(pcap_file, [arp_request, arp_reply])
print(f"[+] ARP diyalog akışı '{pcap_file}' dosyasına başarıyla kaydedildi.\n")

# Dosyadan okuyup doğrulamayı ekrana basalım
paketler = rdpcap(pcap_file)

print("=" * 65)
print("ARP PAKET DÖKÜMÜ VE KATMAN 2 İNCELEMESİ")
print("=" * 65)
for idx, p in enumerate(paketler, start=1):
    op_tipi = "ARP İSTEK (Who-has?)" if p[ARP].op == 1 else "ARP YANIT (Is-at!)"
    print(f"PAKET #{idx} [{op_tipi}]")
    print(f"  [Ethernet] Kaynak MAC: {p[Ether].src} -&gt; Hedef MAC: {p[Ether].dst}")
    print(f"  [ARP Yükü] Gönderen : IP {p[ARP].psrc} (MAC: {p[ARP].hwsrc})")
    print(f"  [ARP Yükü] Hedef    : IP {p[ARP].pdst} (MAC: {p[ARP].hwdst})")
    print("-" * 65)

---
### 4.2. Deney 2'nin Mühendislik Yorumu
1. **Hedef Donanım Adresi Sıfırdır:** İstek paketinde `hwdst` alanı `00:00:00:00:00:00` olarak ayarlanır çünkü sorgulayan makine bu adresi henüz bilmemektedir; öğrenmek istediği bilgi zaten budur.
2. **EtherType Farkı:** Çerçeve başlığındaki `Type` alanı IP paketlerinde `0x0800` iken, ARP paketlerinde **`0x0806`** değerini alır. Ağ kartı bu bayta bakarak gelen yükü IP protokolüne değil, doğrudan işletim sisteminin ARP modülüne teslim eder.

---
## 5. Deney 3: Katman 2 Ağ Anahtarı (Switch) Kendi Kendine Öğrenme (Self-Learning) Simülatörü

### 5.1. Deneyin Amacı
Ders notlarınızda vurgulanan; bir Switch'in IP adresleriyle hiç ilgilenmediğini, gelen çerçevelerin **Kaynak MAC** adreslerine bakarak port tablosunu (*CAM Table*) dinamik olarak nasıl kurduğunu ve hedef MAC tabloda yoksa tüm portlara basma (**Flooding**) mantığını kodlamaktır.

In [ ]:
# ==============================================================================
# DENEY 3: Katman 2 Switch Kendi Kendine Öğrenme ve CAM Tablosu Motoru
# ==============================================================================

class Layer2Switch:
    def __init__(self, port_sayisi=4):
        self.port_sayisi = port_sayisi
        self.cam_table = {}  # MAC Adresi -> Fiziksel Port Numarası

    def process_frame(self, incoming_port: int, src_mac: str, dst_mac: str):
        # 1. KENDİ KENDİNE ÖĞRENME (Self-Learning):
        # Switch gelen çerçevenin KAYNAK MAC adresine bakar. "Bu MAC şu porttan geldi" diye kaydeder.
        if src_mac not in self.cam_table or self.cam_table[src_mac] != incoming_port:
            self.cam_table[src_mac] = incoming_port
            ogrenme_notu = f"[ÖĞRENDİ] {src_mac} adresi Port {incoming_port}'e kaydedildi."
        else:
            ogrenme_notu = f"[BİLİNİYOR] {src_mac} zaten Port {incoming_port}'te kayıtlı."

        # 2. İLETİM (Forwarding) VEYA TAŞIRMA (Flooding) KARARI:
        # Hedef Broadcast ise veya tabloda yoksa gelen port hariç herkese basılır
        if dst_mac == "ff:ff:ff:ff:ff:ff":
            hedef_portlar = [p for p in range(1, self.port_sayisi + 1) if p != incoming_port]
            aksiyon = f"BROADCAST -&gt; Port {incoming_port} hariç tüm portlara fırlatıldı: {hedef_portlar}"
        elif dst_mac in self.cam_table:
            hedef_port = self.cam_table[dst_mac]
            aksiyon = f"DOĞRUDAN İLETİM (Filtreleme) -&gt; Yalnızca Port {hedef_port}'e basıldı."
        else:
            hedef_portlar = [p for p in range(1, self.port_sayisi + 1) if p != incoming_port]
            aksiyon = f"HEDEF TABLODA YOK (FLOODING) -&gt; Tüm portlara taşırıldı: {hedef_portlar}"

        return ogrenme_notu, aksiyon

# --- TEST AKIŞI ---
switch = Layer2Switch(port_sayisi=4)

print("=" * 70)
print("KATMAN 2 SWITCH MAC ÖĞRENME VE İLETİM SİMÜLASYONU")
print("=" * 70)

olaylar = [
    # 1. Host A (Port 1), Host B'ye çerçeve yolluyor (Tablo bomboş)
    (1, "00:00:00:00:00:0A", "00:00:00:00:00:0B"),
    # 2. Host B (Port 2), Host A'ya yanıt veriyor
    (2, "00:00:00:00:00:0B", "00:00:00:00:00:0A"),
    # 3. Host C (Port 3), Host A'ya veri yolluyor
    (3, "00:00:00:00:00:0C", "00:00:00:00:00:0A"),
    # 4. Host A (Port 1), Broadcast (ARP İsteği) yolluyor
    (1, "00:00:00:00:00:0A", "ff:ff:ff:ff:ff:ff")
]

for adim, (port, src, dst) in enumerate(olaylar, start=1):
    not_ogren, not_ilet = switch.process_frame(port, src, dst)
    print(f"ADIM #{adim}: Port {port} -&gt; Gelen Çerçeve [{src} -&gt; {dst}]")
    print(f"  * Tablo Durumu   : {not_ogren}")
    print(f"  * İletim Kararı  : {not_ilet}")
    print("-" * 70)

print("SON CAM TABLOSU (Switch Hafızası):")
for mac, prt in switch.cam_table.items():
    print(f"  - MAC: {mac} ==&gt; Port {prt}")
print("=" * 70)

---
### 5.2. Deney 3'ün Mühendislik Yorumu
1. **İlk Pakette Taşma (Flooding):** 1. adımda Switch Host B'nin nerede olduğunu bilmediği için çerçeveyi tüm portlara basmıştır. Ancak 2. adımda Host B yanıt verdiğinde, Switch Host A'nın Port 1'de olduğunu daha önce öğrendiği için yanıtı tüm ağa saçmamış; **yalnızca Port 1'e** doğrudan iletmiştir.
2. **Tak-Çalıştır (Plug-and-Play) Üstünlüğü:** Switch bir ağ yöneticisinin elle konfigürasyon yapmasını gerektirmez. Cihazlar konuştukça switch kendi tablosunu dinamik doldurur. Cihaz kablosunu çıkarıp başka porta takarsa, ilk iletiminde tablo otomatik güncellenir.

---
## 6. Laboratuvar Uygulaması: ARP Önbelleği Analizi ve ARP Zehirlenmesi (ARP Spoofing / MITM) Güvenlik Zafiyeti

### 6.1. Uygulamanın Amacı
Ders notlarınızda vurgulanan *"Siber güvenliğin temeli ağ protokollerini kod seviyesinde kavramaktır"* ilkesini somutlaştırmaktır. ARP protokolünün hiçbir kimlik doğrulama (*Authentication*) içermemesinden kaynaklanan ve yerel ağ trafiğini saldırganın üzerine çeken **ARP Zehirlenmesi (ARP Cache Poisoning / Spoofing)** zafiyetini simüle etmek ve işletim sistemi ARP tablosunun nasıl manipüle edildiğini analiz etmektir.

### 6.2. Yönerge
1. Bir kurban makinenin yerel işletim sistemi ARP tablosunu (`Kurban_ARP_Cache`) modelleyin.
2. Normal durumda kurban makinenin ağ geçidini (`192.168.1.1` $\rightarrow$ `00:11:22:33:44:01`) bildiğini varsayın.
3. Kurbandan hiçbir istek gelmediği halde saldırganın sahte bir **Gratuitous ARP (Sahte Cevap)** paketi göndererek *"192.168.1.1 benim ve MAC adresim 66:66:66:66:66:66'dır"* yalanını fırlatmasını sağlayın.
4. Kurbanın ARP tablosunun zehirlendiğini ve internete giden tüm paketlerin saldırganın üzerinden geçmeye başladığını (Ortadaki Adam - MITM) konsola formatlı basın.

In [ ]:
# ==============================================================================
# UYGULAMA: ARP Zehirlenmesi (ARP Cache Poisoning / Spoofing) Simülatörü
# ==============================================================================

class HostARPCache:
    def __init__(self, host_name: str, host_ip: str, host_mac: str):
        self.name = host_name
        self.ip = host_ip
        self.mac = host_mac
        self.arp_table = {}  # IP -> MAC

    def receive_arp_packet(self, sender_ip: str, sender_mac: str, is_reply=True):
        """
        Güvenlik Açığı Mantığı: ARP durum tutmaz (Stateless). Bir cihaz sorgu atmamış olsa bile
        gelen bir ARP yanıtına inanır ve tablosunu günceller!
        """
        eski_mac = self.arp_table.get(sender_ip, "YOK (Kayıtsız)")
        self.arp_table[sender_ip] = sender_mac
        return eski_mac, sender_mac

# --- SİMÜLASYON KURULUMU ---
kurban_pc = HostARPCache("Kurban PC", "192.168.1.50", "AA:AA:AA:50:50:50")

# 1. Aşama: Masum Başlangıç (Kurban modemi tanıyor)
GERCEK_GATEWAY_IP = "192.168.1.1"
GERCEK_GATEWAY_MAC = "00:11:22:01:01:01"
kurban_pc.receive_arp_packet(GERCEK_GATEWAY_IP, GERCEK_GATEWAY_MAC)

print("=" * 70)
print("ARP CACHE POISONING (ZEHİRLENME) GÜVENLİK ANALİZİ")
print("=" * 70)
print(f"[BAŞLANGIÇ] Kurbanın Güvenli ARP Tablosu:")
print(f"  -> Gateway IP: {GERCEK_GATEWAY_IP}  ==&gt;  MAC: {kurban_pc.arp_table[GERCEK_GATEWAY_IP]} (Gerçek Modem)")
print("-" * 70)

# 2. Aşama: Saldırgan Ağda Sahte ARP Paketi Fırlatıyor (Gratuitous ARP)
SALDIRGAN_MAC = "66:66:66:BA:D0:00"
print("[SALDIRI] Kötü niyetli cihaz ağa sahte ARP cevabı basıyor:")
print(f"  -> '192.168.1.1 benim ve MAC adresim {SALDIRGAN_MAC}!'")

# Kurban bu paketi sorgulamadan alır
eski, yeni = kurban_pc.receive_arp_packet(GERCEK_GATEWAY_IP, SALDIRGAN_MAC)

print("\n[SONUÇ] Kurbanın Zehirlenmiş Güncel ARP Tablosu:")
print(f"  -> Gateway IP: {GERCEK_GATEWAY_IP}  ==&gt;  MAC: {kurban_pc.arp_table[GERCEK_GATEWAY_IP]} (SALDIRGANIN MAC'İ!)")
print("=" * 70)

# 3. Aşama: Kurban İnternete Veri Gönderirken Ne Olur?
print("MÜHENDİSLİK VE GÜVENLİK ETKİSİ:")
print("Kurban PC Google'a bir şifre veya e-posta gönderirken paketin hedef IP'sini 8.8.8.8 yapar,")
print(f"ancak çerçevenin hedef MAC adresine tablodan bakıp '{kurban_pc.arp_table[GERCEK_GATEWAY_IP]}' yazar.")
print("Katman 2 Switch bu çerçeveyi doğrudan saldırganın bilgisayarına teslim eder!")
print("Saldırgan tüm trafiği okur, kaydeder ve sonra modeme iletir (Man-in-the-Middle - MITM).")
print("=" * 70)

---
### 6.3. Konsol Çıktısı ve Mühendislik Yorumu

```text
======================================================================
ARP CACHE POISONING (ZEHİRLENME) GÜVENLİK ANALİZİ
======================================================================
[BAŞLANGIÇ] Kurbanın Güvenli ARP Tablosu:
  -> Gateway IP: 192.168.1.1  ==>  MAC: 00:11:22:01:01:01 (Gerçek Modem)
----------------------------------------------------------------------
[SALDIRI] Kötü niyetli cihaz ağa sahte ARP cevabı basıyor:
  -> '192.168.1.1 benim ve MAC adresim 66:66:66:BA:D0:00!'

[SONUÇ] Kurbanın Zehirlenmiş Güncel ARP Tablosu:
  -> Gateway IP: 192.168.1.1  ==>  MAC: 66:66:66:BA:D0:00 (SALDIRGANIN MAC'İ!)
======================================================================
MÜHENDİSLİK VE GÜVENLİK ETKİSİ:
Kurban PC Google'a bir şifre veya e-posta gönderirken paketin hedef IP'sini 8.8.8.8 yapar,
ancak çerçevenin hedef MAC adresine tablodan bakıp '66:66:66:BA:D0:00' yazar.
Katman 2 Switch bu çerçeveyi doğrudan saldırganın bilgisayarına teslim eder!
Saldırgan tüm trafiği okur, kaydeder ve sonra modeme iletir (Man-in-the-Middle - MITM).
======================================================================
```

#### Mühendislik Yorumu:
1. **ARP'nin Durumsuzluk (Stateless) Zafiyeti:** ARP protokolünde bir güvenlik doğrulaması (*Authentication*) yoktur. Bir cihaz kendisine sahte bir `ARP Reply` geldiğinde, *"Ben bunu hiç sormamıştım, neden cevap geldi?"* diye sorgulamaz; gelen bilgiyi doğrudan RAM'indeki tabloya yazar. Güvenliksiz tasarlanmış eski yerel ağ protokollerinin en kırılgan noktası budur.
2. **Savunma Mekanizmaları:** Modern kurumsal ağlarda bu saldırıyı engellemek için akıllı Switch'ler üzerinde **DAI (Dynamic ARP Inspection)** ve **DHCP Snooping** özellikleri devreye alınır; switch yalnızca DHCP sunucusunun dağıttığı meşru IP-MAC çiftlerine ait ARP paketlerinin geçişine izin verir.